In [0]:
from datetime import datetime
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "dev")
CURRENT_CATALOG = dbutils.widgets.get("catalog")

landing_root = f"/Volumes/{CURRENT_CATALOG}/kruger/landing/raw_youtube"
staging_root = f"/Volumes/{CURRENT_CATALOG}/kruger/staging/filtered_youtube"

print(f"Processing latest YouTube batch from {landing_root} into {staging_root}")

try:
    # 1. Grab ONLY the newest JSON file in landing_root
    files = [f for f in dbutils.fs.ls(landing_root) if f.name.endswith(".json")]
    if not files:
        raise FileNotFoundError("No JSON files found in landing folder.")
        
    latest_file = max(files, key=lambda f: f.modificationTime).path
    print(f"Reading latest file: {latest_file}")

    # 2. Filter only the 8 columns you want
    df_raw = spark.read.option("multiline", "true").json(latest_file)
    
    df_filtered = df_raw.select(F.explode("items").alias("video")).select(
        F.col("video.snippet.title").alias("title"),
        F.col("video.id").alias("id"),
        F.col("video.snippet.publishedAt").alias("published_at"),
        F.col("video.snippet.liveBroadcastContent").alias("live_broadcast_content"),
        F.col("video.statistics.viewCount").alias("view_count"),
        F.col("video.statistics.commentCount").alias("comment_count"),
        F.col("video.statistics.favoriteCount").alias("favorite_count"),
        F.col("video.statistics.likeCount").alias("like_count")
    )
    
    # 3. Write to a unique timestamped subfolder (never overwrite!)
    run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    batch_output_path = f"{staging_root}/batch_{run_ts}"
    df_filtered.write.mode("append").json(batch_output_path)
    
    count = df_filtered.count()
    print("-" * 45)
    print(f"{'source':<20}{'format':<8}{'batch_rows'}")
    print("-" * 45)
    print(f"{'latest_sightings':<20}{'json':<8}{count}")

except Exception as e:
    print(f"Error processing batch: {e}")
